<h1>Gaia Star Cluster Hertzsprung Russel Diagrams (HRD)</h1>

Here are some useful links 
- [European Space Agency Gaia Mission - Writing Queries Turorial](https://www.cosmos.esa.int/web/gaia-users/archive/writing-queries)
- [Gaia's Hertzsprung-Russel Diagram](https://sci.esa.int/web/gaia/-/60198-gaia-hertzsprung-russell-diagram)
- [Measuring the Age of a Star Cluster](https://www.e-education.psu.edu/astro801/content/l7_p6.html#:~:text=The%20HR%20diagram%20for%20stage,%2D13%20billion%20years%20old)
- [Astro Prof YouTube Channel Phys 1403](https://www.youtube.com/watch?v=sDds6c7HByg)

**In the examples below we will query open clusters in the Milky way and plot it in a Hertzsprung-Russel Diagram (HRD). We will also attempt to identify the Main Sequence Turnoff points for these star clusters**

*adapted for the Python for Astronomy Course by Chandru Narayan using the material originally develped by Dr. Priya Hasan and the Gaia utilities*

# Steps to create Hertzsprung-Russel Diagram

- Step 1:  Install the required libraries for astronomy data analysis, querying, and plotting.
- Step 2:  Import the necessary modules from the installed libraries.
- Step 3:  Query the Gaia archive for data within a specified radius around a given cluster.
- Step 4a: Create a Dataframe and Cleanup data as needed.
- Step 4b: Calculate absolute magnitudes and colors from the Gaia data.
- Step 5:  Generate HR diagrams, plotting absolute magnitude against color.
- Repeat Steps 3-5 above for another Star Cluster

## Step 1: Install necessary libraries as necessary

In [ ]:
# pip install astropy astroquery matplotlib google

## Step 2: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia
import astropy.coordinates as coord
import pandas as pd

## Step 3: Query Gaia data

### First perform a Google Search about your target

In [ ]:
### Setup your target here
target_id = 'M67'
# provide approximate target distance in parsec and target size in minutes for Gaia query
# change default values below
target_dist = 850 # in parsec
target_size = 60 # in minutes

In [ ]:
try:
	from googlesearch import search
except ImportError:
	print("No module named 'google' found")

# to search
query = f'{target_id} Wikipedia Astronomy'

for j in search(query, tld="co.in", num=10, stop=10, pause=2):
	print(j)


In [ ]:
%%html
<div style="text-align:center;">
<iframe src="https://gea.esac.esa.int/archive/" width="900" height="540"></iframe>
</div>

### Perform your Query

In [ ]:
# Parameters for API Query & HRD (20 minutes radius around Cluster center) - Change for each query!
target = target_id # target to query
object_radius = target_size/60 * u.deg # in deg
coordinate = coord.SkyCoord.from_name(target)
print(coordinate)

# Define cluster coordinates and radius for Cone search
radius = object_radius
ra = coordinate.ra
dec = coordinate.dec


In [ ]:
# Query strings for Gaia
# Query 1
query1 = f"""
    SELECT ra, dec, parallax, 1000/parallax as dist, phot_g_mean_mag, bp_rp
    FROM gaiaedr3.gaia_source 
    WHERE parallax > 0 
    AND 1=CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra.value}, {dec.value}, {radius.value})
        )
"""
# Query 2
query2 = f"""
    SELECT ra, dec, parallax, 1000/parallax as dist, phot_g_mean_mag, bp_rp
    FROM gaiaedr3.gaia_source 
    WHERE parallax > 0 
    AND bp_rp > -0.75
    AND bp_rp < 6
    AND visibility_periods_used > 8
    AND phot_g_mean_flux_over_error > 50
    AND phot_bp_mean_flux_over_error > 20
    AND phot_rp_mean_flux_over_error > 20
    and phot_bp_rp_excess_factor <
    1.3+0.06*power(phot_bp_mean_mag-phot_rp_mean_mag,2)
    and phot_bp_rp_excess_factor >
    1.0+0.015*power(phot_bp_mean_mag-phot_rp_mean_mag,2)
    and astrometric_chi2_al/(astrometric_n_good_obs_al-5)<
    1.44*greatest(1,exp(-0.4*(phot_g_mean_mag-19.5)))
    AND 1=CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {ra.value}, {dec.value}, {radius.value})
        )
""" 

In [ ]:
# create and launch async query
job = Gaia.launch_job_async(query2)
gaia_data = job.get_results()
#print(gaia_data)

## Step 4a: Create a Data Frame & Cleanup Gaia Query Results data

In [ ]:
# Create a DataFrame and Cleanup specific to your target:
allstars = gaia_data.to_pandas()
print("allstars\n",allstars)
# filter allstars
allstars.dropna(inplace=True)  # Drop NaN - "not a number"
print("allstars_cleaned\n",allstars)
# filter for dist between 800 & 900 pc (allstars_filtered) by creating a mask
#mask = (allstars.dist >= 800) & (allstars.dist <= 900) & (allstars.bp_rp >= 0.7) & (allstars.bp_rp <= 0.8)
tgt_min_dist = target_dist * 0.9 # create a range
tgt_max_dist = target_dist * 1.1 # create a range
mask = (allstars.dist >= tgt_min_dist) & (allstars.dist <= tgt_max_dist)
allstars = allstars[mask]
print("allstars_masked\n",allstars)

## Step 4b: Calculate absolute magnitude and add to DataFrame

### Review Formulas (we drived these in the Star MAgnitudes Module!)
#### The quantity $\boxed{m_{app} - m_{abs}} $ OR $ \boxed {m - M} $ is known as the distance modulus
#### Note that this quantity appears in the equations below to calculate magnitudes and distance
### 1. How to calculate Magnitudes when distance (in pc) is known
#### $$ \boxed{m - M =  5 \times log_{10}(distance) - 5} $$
### 2. How to calculate Magnitudes when parallax (in arc-sec) is known
#### $$ \boxed{m - M =  5 \times log_{10}(1/parallax) - 5} $$
### 3. How to calculate Magnitudes for Gaia calculations when parallax (in milli-arc-sec or 'mas') is known
##### $$ \boxed{m -M = - 5 \times log_{10}(parallax) + 10} $$ OR $$ \boxed{M = m + 5 \times log_{10}(parallax) - 10} $$
#### 4. How to calculate Distance (in pc) when apparent and absolute magnitudes are known
##### $$ \boxed{distance = 10^{\frac{m - M+5}{5}}} $$

In [ ]:
# Calculate absolute magnitude with parallax in mas
# Make sure parallax > 0 by creating another filter

distance_modulus = -5 * np.log10(allstars['parallax']) + 10
abs_mag = allstars['phot_g_mean_mag'] - distance_modulus
bp_rp = allstars['bp_rp']

## Step 5: Plot HR diagram

### Simple Plot of HRD with the Sun superimposed

In [ ]:
# sun's HRD coordinates values for plotting
#     google "what is bp-rp of the sun" bp-rp = 0.82
bp_rp_sun = 0.82

#     we calculated the abs mag of the sun in the "brightness of stars" project
#     we will do it again below:
sun_app_mag = -27 # very very bright in the sky!
sun_dist_km = 150e6 #km
light_speed = 3e5 #km/s
seconds_in_a_year = 365*24*60*60
light_year_km =  seconds_in_a_year * light_speed
parsec_km = light_year_km * 3.26 #km/pc
sun_dist_pc = sun_dist_km / parsec_km

# Use the magnitude formula above to calculate the absolute magnitude of the Sun
sun_abs_mag = sun_app_mag - 5 * np.log10(sun_dist_pc) + 5
#print(sun_abs_mag)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(bp_rp, abs_mag, s=1, alpha=0.5) # plot stars
plt.scatter(bp_rp_sun, sun_abs_mag, marker='+', color="red") # plot sun
plt.text(bp_rp_sun+0.2, sun_abs_mag, "Red Cross is our Sun!",  color="red") # mark sun
plt.gca().invert_yaxis()
plt.xlabel("BP-RP Color")
plt.ylabel("Absolute G Magnitude")
plt.title(f"HR Diagram for {target}")
plt.savefig(f"{target}.png", dpi=140)
plt.show()

## Can you tell where the Main Sequence Turnoff is happening?
## What does that tell you about the Age of the cluster?
![](age_turnoff_point.png)
## Answer Questions below

In [ ]:
# Write print statement below with your answer
abs_mag_turnoff = 4.0
turnoff_age = 7.1e9
print(f'Main Sequence cutoff for {target} is at Abs Mag {abs_mag_turnoff}')
print(f'This indicates that {target} is {turnoff_age:.1e} years old!')

# Repeat Steps 3 to 5 below for another Star Cluster